In [96]:
import torch 
import torchvision 
import torch.nn as nn 
import torch.nn.functional as F  
from torch.utils.data import Subset
import torchvision.transforms.v2 as v2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms.functional as TF

import numpy as np 
import pandas as pd 
from tqdm import tqdm 
import matplotlib as mpl 
import matplotlib.pyplot as plt 
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from scipy.stats import norm

import os
import glob 
import logging
import datetime
from os import cpu_count
from string import hexdigits
from multiprocessing import Manager
from multiprocessing.dummy import Pool

IMAGE_SIZE = 320

df = pd.read_csv('./malware-dataset.csv')

In [97]:
def init_shared_dict(shared_dict_):
    global shared_dict
    shared_dict = shared_dict_ 

def read_size(args):
    path, label = args 
    shared_dict[path] = os.path.getsize(path)

if not os.path.exists("size.csv"):
    paths = df['0']
    labels = df['1']

    paths_labels = [tuple(x) for x in np.array([paths, labels]).T.tolist()]
    with Manager() as m: 
        shared_dict = m.dict() 
        with Pool(processes=cpu_count(), initializer=init_shared_dict, initargs=(shared_dict,)) as pool:
            list(tqdm(pool.imap_unordered(read_size, paths_labels), total=len(paths_labels)))
        size_dict = dict(shared_dict)
        size_dict = pd.DataFrame({"path": list(size_dict.keys()), "size": list(size_dict.values()), "label": [l for _, l in paths_labels]})
        size_dict.to_csv("size.csv")
else: 
    size_dict = pd.read_csv("size.csv")

In [98]:
def calculate_sf(size):
    shape = np.ceil(np.sqrt(size)).astype(np.int32)
    return (shape ** 2) / (320 ** 2)

sfs = []
for _, _,_,  size, label in size_dict.itertuples():
    sfs.append(calculate_sf(size))
size_dict.insert(len(size_dict.columns), 'factor', sfs)

In [99]:
def get_rows(label):
    return size_dict[size_dict['label'] == label]

benign = get_rows(0)
malicious = get_rows(1)

In [100]:
mean_m, std_m = norm.fit(malicious['factor'])
mean_b, std_b = norm.fit(benign['factor'])

print(f"Malicious:\tN({mean_m:.2f}, {std_m:.2f})")
print(f"Benign:\t\tN({mean_b:.2f}, {std_b:.2f})")

Malicious:	N(3.51, 5.16)
Benign:		N(54.56, 396.94)


In [109]:
ub = mean_m + (2.5 * std_m)
lb = mean_m - (2.5 * std_m)

print(f"{lb} <= sf <= {ub}")
valid_benign = benign[(benign['factor'] >= lb) & (benign['factor'] <= ub)]
valid_malicious = malicious[(malicious['factor'] >= lb) & (malicious['factor'] <= ub)]
valid = pd.concat([valid_malicious, valid_benign])
valid.to_csv('valid_dataset.csv')

-9.391114527772608 <= sf <= 16.414383911324276


In [108]:
valid['factor'].describe()

count    10155.000000
mean         2.706763
std          3.221175
min          0.000000
25%          0.481289
50%          1.485352
75%          3.621885
max         18.976914
Name: factor, dtype: float64

In [124]:
lut, ds, size = map(pd.read_csv, ['lookup.csv', 'malware-dataset.csv', 'size.csv'])
tmp = pd.merge(lut, ds, left_on='original', right_on='0').drop(columns=['Unnamed: 0_x', 'Unnamed: 0_y', '0'])
tmp.rename(columns={'1': 'label'})
tmp = pd.merge(tmp, size, left_on='original', right_on='path').drop(columns=['Unnamed: 0', '1', 'path'])
tmp.to_csv('dataset.csv')